# Hospital Patient Profiling
## Unsupervised Machine Learning for Patient Segmentation
**Dataset**: Kaggle ICU Patient Dataset (`train.csv`)
**Algorithm**: K-Means Clustering + PCA Visualization


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import joblib


### 1. Load and Inspect Dataset


In [ ]:
path = '../data/train.csv' if os.path.exists('../data/train.csv') else 'data/train.csv'
df = pd.read_csv(path)
print('Dataset Shape:', df.shape)
display(df.head())
print('\nMissing Values:')
print(df.isnull().sum())


### 2. Feature Selection & Preprocessing Pipeline


In [ ]:
feature_cols = ['age', 'gender', 'bmi', 'heart_rate', 'systolic_bp', 'diastolic_bp', 'respiratory_rate', 'oxygen_saturation', 'glucose', 'creatinine', 'wbc_count', 'severity_score', 'ventilation_required', 'chronic_conditions']
num_cols = [c for c in feature_cols if df[c].dtype in ['int64', 'float64']]
pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
X_scaled = pipeline.fit_transform(df[feature_cols])
print('Processed Feature Matrix Shape:', X_scaled.shape)


### 3. K-Means Clustering & Optimal K Evaluation


In [ ]:
k_results = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    k_results.append({'K': k, 'Inertia': km.inertia_, 'Silhouette': sil})
k_df = pd.DataFrame(k_results)
display(k_df)


### 4. Train Final Model (K=4)


In [ ]:
final_model = KMeans(n_clusters=4, random_state=42, n_init='auto')
cluster_labels = final_model.fit_predict(X_scaled)
df_imputed = df.copy()
df_imputed[num_cols] = SimpleImputer(strategy='median').fit_transform(df[num_cols])
df_imputed['cluster'] = cluster_labels
print('Cluster Sizes:')
print(df_imputed['cluster'].value_counts())


### 5. PCA 2D Projection


In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)
plt.figure(figsize=(8, 6))
sns.scatterplot(x=pca_coords[:,0], y=pca_coords[:,1], hue=cluster_labels, palette='tab10', alpha=0.7)
plt.title('2D PCA Projection of Patient Clusters')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.show()


### 6. Cluster Profiles & Model Persistence


In [ ]:
profile_stats = df_imputed.groupby('cluster')[feature_cols + ['survived_icu']].mean()
display(profile_stats)
os.makedirs('../models', exist_ok=True)
joblib.dump(final_model, '../models/clustering_model.joblib')
joblib.dump(pipeline, '../models/preprocessing_pipeline.joblib')
print('Model artifacts successfully saved!')
